# CREEDS Gene — pert_type & chem-duplicate review

In [1]:
import gzip, json, re, uuid
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
from maayanlab_bioinformatics.parse import gmt_read_iter

pd.set_option('display.max_rows', None)      # never truncate rows
pd.set_option('display.max_colwidth', 120)

MAIN = Path('/Users/johngardner_mountsinai/work/PerturbSeqr')
GENE_GMT = MAIN / 'db/data/gene_perturbation/creeds-gene.gmt.gz'
CHEM_GMT = MAIN / 'db/data/drug_perturbation/creeds-chem.gmt.gz'
CACHE = MAIN / 'db/etl/inputs/_cache/creeds-gene.json'

def load_gmt(path):  # gzip adapter around maayanlab_bioinformatics' GMT parser
    with gzip.open(path, 'rt') as fh:
        return [(term, list(geneset)) for term, geneset in gmt_read_iter(fh)]

def gs_hash(genes):  # the project-specific hash the ingest uses (db/ingest.py)
    return str(uuid.uuid5(uuid.UUID(int=0), '\t'.join(sorted(set(genes)))))

def gse_of(term):
    m = re.search(r'GSE\d+', term)
    return m.group(0) if m else ''

gene = load_gmt(GENE_GMT)
chem = load_gmt(CHEM_GMT)
raw = json.loads(CACHE.read_text())
raw_pert = {e.get('id', '').split(':')[-1]: e.get('pert_type', '') for e in raw}
print(f'CREEDS gene terms: {len(gene)}   chem terms: {len(chem)}   raw metadata entries: {len(raw)}')

CREEDS gene terms: 4778   chem terms: 1542   raw metadata entries: 2176


## 1. Duplicates: CREEDS Gene ↔ CREEDS Chem (byte-identical gene sets)

In [2]:
chem_by_hash = defaultdict(list)
for t, g in chem:
    chem_by_hash[gs_hash(g)].append(t)

dup_rows = []
for t, g in gene:
    twins = chem_by_hash.get(gs_hash(g))
    if not twins:
        continue
    body, direction = t.rsplit(' ', 1)
    f = body.split('::')
    creeds_id = f[8].split('.')[-1] if len(f) > 8 and '.' in f[8] else ''
    pt = raw_pert.get(creeds_id, '')
    for chem_term in twins:
        dup_rows.append({
            'gse': gse_of(t) or gse_of(chem_term),
            'raw_pert_type': pt,
            'gene_term': t,
            'chem_term': chem_term,
            'direction': direction,
            'n_genes': len(g),
        })

dup_df = pd.DataFrame(dup_rows).sort_values(['gse', 'gene_term']).reset_index(drop=True)
print(f'{len(dup_df)} duplicate terms ({dup_df["gene_term"].str.rsplit(" ", n=1).str[0].nunique()} signatures)')
with pd.option_context('display.max_rows', None):
    display(dup_df)

60 duplicate terms (30 signatures)


,gse,raw_pert_type,gene_term,chem_term,direction,n_genes
0,GSE11237,INHIBITION,PTGS2::::Colorectal adenocarcinoma::::::::::homo sapiens::GSE11237.1037 down,Celecoxib::::Primary Colorectal Adenocarcinoma::::::homo sapiens::GSE11237.2458 down,down,272
1,GSE11237,INHIBITION,PTGS2::::Colorectal adenocarcinoma::::::::::homo sapiens::GSE11237.1037 up,Celecoxib::::Primary Colorectal Adenocarcinoma::::::homo sapiens::GSE11237.2458 up,up,311
2,GSE11670,Repression,DDIT4::::Blood Myeloid Cells::::::KD::::homo sapiens::GSE11670.100 down,Deferasirox::::K562 (human myeloid leukemia cells)::::::homo sapiens::GSE11670.2571 down,down,327
3,GSE11670,Repression,DDIT4::::Blood Myeloid Cells::::::KD::::homo sapiens::GSE11670.100 up,Deferasirox::::K562 (human myeloid leukemia cells)::::::homo sapiens::GSE11670.2571 up,up,235
4,GSE19736,INHIBITION,GSK3B::::MLL leukemia cells::::::::::homo sapiens::GSE19736.486 down,Sb 216763::::RS4.11 leukemia cell line::::::homo sapiens::GSE19736.3266 down,down,273
5,GSE19736,INHIBITION,GSK3B::::MLL leukemia cells::::::::::homo sapiens::GSE19736.486 up,Sb 216763::::RS4.11 leukemia cell line::::::homo sapiens::GSE19736.3266 up,up,237
6,GSE22288,drugactivation,ERBB3::::MCF10::::::OE::::homo sapiens::GSE22288.1877 down,Heregulin::::Mammary epithelial cells (MCF10A/HER2/FKBP/HA cells)::::::homo sapiens::GSE22288.3215 down,down,321
7,GSE22288,drugactivation,ERBB3::::MCF10::::::OE::::homo sapiens::GSE22288.1877 up,Heregulin::::Mammary epithelial cells (MCF10A/HER2/FKBP/HA cells)::::::homo sapiens::GSE22288.3215 up,up,230
8,GSE24493,druginhibition,"ABL1::::chronic myelogenous leukemia cell lines KU-812, KCL-22 and JURL-MK1::::::::::homo sapiens::GSE24493.1904 down",Imatinib::::NA::::::homo sapiens::GSE24493.2764 down,down,394
9,GSE24493,druginhibition,"ABL1::::chronic myelogenous leukemia cell lines KU-812, KCL-22 and JURL-MK1::::::::::homo sapiens::GSE24493.1904 up",Imatinib::::NA::::::homo sapiens::GSE24493.2764 up,up,182


## 2. Every CREEDS Gene `pert_type`, by frequency

In [3]:
counts = Counter(e.get('pert_type', '') for e in raw)
pt_df = pd.DataFrame(counts.most_common(), columns=['pert_type', 'count'])
print(f'{len(pt_df)} distinct pert_type values across {pt_df["count"].sum()} entries')
with pd.option_context('display.max_rows', None):
    display(pt_df)

260 distinct pert_type values across 2176 entries


,pert_type,count
0,KO,752
1,KD,499
2,OE,244
3,Deficiency,116
4,knockout,43
5,DEPLETION,29
6,knockdown,27
7,druginhibition,25
8,activemutant,15
9,Knock-in,10
